# Notebook 1: Fetch NBA Data
Pulls 2025 NBA player box score data using `nba_api` and saves it as CSV files.

In [1]:

import pandas as pd
import time
import os
from nba_api.stats.endpoints import leaguegamelog, boxscoretraditionalv3
from nba_api.stats.static import players, teams
import random
import json

os.makedirs('data', exist_ok=True)
print('Libraries loaded.')

Libraries loaded.


## Step 1: Fetch All Games for 2024-25 Season

In [2]:
# Fetch regular season game log
gamelog_rs = leaguegamelog.LeagueGameLog(
    season='2024-25',
    season_type_all_star='Regular Season'
).get_data_frames()[0]

# Fetch playoffs game log
gamelog_po = leaguegamelog.LeagueGameLog(
    season='2024-25',
    season_type_all_star='Playoffs'
).get_data_frames()[0]

# Combine and tag season type
gamelog_rs['SEASON_TYPE'] = 'Regular Season'
gamelog_po['SEASON_TYPE'] = 'Playoffs'
all_games = pd.concat([gamelog_rs, gamelog_po], ignore_index=True)

print(f'Total team-game records: {len(all_games)}')
all_games.head()

Total team-game records: 2628


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE,SEASON_TYPE
0,22024,1610612738,BOS,Boston Celtics,0022400061,2024-10-22,BOS vs. NYK,W,240,48,...,40,33,6,3,4,15,132,23,1,Regular Season
1,22024,1610612750,MIN,Minnesota Timberwolves,0022400062,2024-10-22,MIN @ LAL,L,240,35,...,47,17,4,1,16,22,103,-7,1,Regular Season
2,22024,1610612747,LAL,Los Angeles Lakers,0022400062,2024-10-22,LAL vs. MIN,W,240,42,...,46,22,7,8,7,22,110,7,1,Regular Season
3,22024,1610612752,NYK,New York Knicks,0022400061,2024-10-22,NYK @ BOS,L,240,43,...,34,20,2,3,12,12,109,-23,1,Regular Season
4,22024,1610612744,GSW,Golden State Warriors,0022400072,2024-10-23,GSW @ POR,W,240,48,...,57,38,13,5,18,27,140,36,1,Regular Season


## Step 2: Build Games Table

In [3]:
# Deduplicate to one row per game
games_df = all_games[['GAME_ID', 'GAME_DATE', 'MATCHUP', 'SEASON_TYPE']].drop_duplicates(subset='GAME_ID').copy()

# Parse home/away from MATCHUP (e.g. 'GSW vs. LAL' or 'GSW @ LAL')
def parse_matchup(row):
    if 'vs.' in row['MATCHUP']:
        home = row['MATCHUP'].split(' vs. ')[0].strip()
        away = row['MATCHUP'].split(' vs. ')[1].strip()
    else:
        away = row['MATCHUP'].split(' @ ')[0].strip()
        home = row['MATCHUP'].split(' @ ')[1].strip()
    return home, away

games_df[['HOME_TEAM', 'AWAY_TEAM']] = games_df.apply(
    lambda r: pd.Series(parse_matchup(r)), axis=1
)

games_df = games_df[['GAME_ID', 'GAME_DATE', 'HOME_TEAM', 'AWAY_TEAM', 'SEASON_TYPE']]
games_df.to_csv('data/games.csv', index=False)
print(f'Games saved: {len(games_df)}')
games_df.head()

Games saved: 1314


,GAME_ID,GAME_DATE,HOME_TEAM,AWAY_TEAM,SEASON_TYPE
0,0022400061,2024-10-22,BOS,NYK,Regular Season
1,0022400062,2024-10-22,LAL,MIN,Regular Season
4,0022400072,2024-10-23,POR,GSW,Regular Season
5,0022400068,2024-10-23,HOU,CHA,Regular Season
6,0022400071,2024-10-23,LAC,PHX,Regular Season


## Step 3: Fetch Player Box Scores
> ⚠️ This step makes one API call per game and includes a short delay to avoid rate limiting. It may take 20–40 minutes for the full season.

In [4]:
# ── Cell 5: Fetch player box scores with checkpointing + retry queue ─────
import os, json
from nba_api.stats.endpoints import boxscoretraditionalv3

CHECKPOINT_DIR = "data/checkpoints"
CHECKPOINT_LOG = "data/checkpoints/progress.json"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def save_chunk(records, chunk_num):
    path = f"{CHECKPOINT_DIR}/chunk_{chunk_num:04d}.csv"
    pd.DataFrame(records).to_csv(path, index=False)
    print(f"  ✓ Saved chunk {chunk_num} ({len(records)} records) → {path}")

def load_progress():
    if os.path.exists(CHECKPOINT_LOG):
        with open(CHECKPOINT_LOG) as f:
            return json.load(f)
    return {"last_completed_index": -1, "retry_queue": [], "permanent_failures": [], "chunk_count": 0}

def save_progress(progress):
    with open(CHECKPOINT_LOG, "w") as f:
        json.dump(progress, f, indent=2)

def fetch_game(gid, retries=3, base_wait=10):
    """Try to fetch a game up to `retries` times with increasing wait."""
    for attempt in range(retries):
        try:
            bs = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id=gid, timeout=60)
            return bs.get_data_frames()[0]
        except Exception as e:
            if attempt < retries - 1:
                wait = base_wait * (attempt + 1)
                print(f"    Attempt {attempt + 1} failed for {gid}: {e} — retrying in {wait}s")
                time.sleep(wait)
            else:
                raise   # re-raise on final attempt so caller can handle it

def flush_chunk(buffer, chunk_num, progress):
    """Concat buffer, save to disk, update progress."""
    chunk_num += 1
    combined = pd.concat(buffer, ignore_index=True)
    save_chunk(combined, chunk_num)
    progress["chunk_count"] = chunk_num
    save_progress(progress)
    return chunk_num, []   # return new chunk_num and empty buffer

# ── Resume ───────────────────────────────────────────────────────────────
progress  = load_progress()
start_idx = progress["last_completed_index"] + 1
chunk_num = progress["chunk_count"]

# Migrate old format if needed
if "failed" in progress and "retry_queue" not in progress:
    progress["retry_queue"]        = progress.pop("failed")
    progress["permanent_failures"] = []

retry_queue        = progress.get("retry_queue", [])
permanent_failures = progress.get("permanent_failures", [])

game_ids = games_df["GAME_ID"].unique().tolist()

# Append any queued retries to the end of the main list
# (only those not already in remaining games)
remaining     = game_ids[start_idx:]
queued_extras = [g for g in retry_queue if g not in remaining]
full_queue    = remaining + queued_extras   # retry games come after new ones

if start_idx > 0 or queued_extras:
    print(f"Resuming from index {start_idx}/{len(game_ids)}")
    print(f"  + {len(queued_extras)} games re-queued from previous failures")
else:
    print(f"Starting fresh — {len(game_ids)} games total")

CHUNK_SIZE    = 200
buffer        = []
newly_failed  = []   # failures in THIS session that haven't been retried yet

for i, gid in enumerate(full_queue):
    is_retry = gid in retry_queue

    try:
        df = fetch_game(gid)   # already retries 3x internally
        buffer.append(df)

        # Only advance last_completed_index for the main list, not re-queued games
        if not is_retry:
            progress["last_completed_index"] = start_idx + i
        
        # Remove from retry_queue if it succeeded
        if gid in retry_queue:
            retry_queue.remove(gid)
            progress["retry_queue"] = retry_queue
            print(f"    ✓ Recovered previously failed game {gid}")

        is_last = (i == len(full_queue) - 1)
        if len(buffer) >= CHUNK_SIZE or (is_last and buffer):
            chunk_num, buffer = flush_chunk(buffer, chunk_num, progress)
            print(f"  Progress: {i + 1}/{len(full_queue)} games done")

        time.sleep(random.randint(1, 5))

    except Exception as e:
        print(f"  ✗ Gave up on {gid} after 3 attempts: {e}")
        newly_failed.append(gid)

        # Push to retry_queue for next run (if not already there)
        if gid not in retry_queue:
            retry_queue.append(gid)
        progress["retry_queue"] = retry_queue
        save_progress(progress)
        time.sleep(random.randint(1, 5))

# Save any remaining buffer
if buffer:
    chunk_num, buffer = flush_chunk(buffer, chunk_num, progress)

# After the full run, anything still in retry_queue becomes a permanent failure
# only if it has failed across multiple runs (you can tune this threshold)
print(f"\nDone. {chunk_num} chunks saved.")
print(f"  Failed this session (re-queued for next run): {len(newly_failed)}")
print(f"  Total in retry queue for next run: {len(retry_queue)}")
if retry_queue:
    print("  Re-queued IDs:", retry_queue)

Resuming from index 1314/1314
  + 0 games re-queued from previous failures

Done. 13 chunks saved.
  Failed this session (re-queued for next run): 0
  Total in retry queue for next run: 0


## Step 4: Build PlayerBoxScores and Players Tables

In [6]:
# Run this first to see actual column names
import glob
chunk_files = sorted(glob.glob(f"{CHECKPOINT_DIR}/chunk_*.csv"))
sample = pd.read_csv(chunk_files[0])
print(sample.columns.tolist())
print(sample.head(2))

['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'personId', 'firstName', 'familyName', 'nameI', 'playerSlug', 'position', 'comment', 'jerseyNum', 'minutes', 'fieldGoalsMade', 'fieldGoalsAttempted', 'fieldGoalsPercentage', 'threePointersMade', 'threePointersAttempted', 'threePointersPercentage', 'freeThrowsMade', 'freeThrowsAttempted', 'freeThrowsPercentage', 'reboundsOffensive', 'reboundsDefensive', 'reboundsTotal', 'assists', 'steals', 'blocks', 'turnovers', 'foulsPersonal', 'points', 'plusMinusPoints']
     gameId      teamId teamCity teamName teamTricode teamSlug  personId  \
0  22400061  1610612738   Boston  Celtics         BOS  celtics   1627759   
1  22400061  1610612738   Boston  Celtics         BOS  celtics   1628369   

  firstName familyName     nameI  ... reboundsOffensive reboundsDefensive  \
0    Jaylen      Brown  J. Brown  ...                 2                 5   
1    Jayson      Tatum  J. Tatum  ...                 0                 4   

  reb

In [5]:
# ── Cell 6: Reassemble chunks into final CSVs ────────────────────────────
import glob

chunk_files = sorted(glob.glob(f"{CHECKPOINT_DIR}/chunk_*.csv"))
print(f"Found {len(chunk_files)} chunk files — combining...")

boxscores_df = pd.concat([pd.read_csv(f) for f in chunk_files], ignore_index=True)

# Cast GAME_ID immediately after concat before any renaming or merging
boxscores_df['gameId'] = boxscores_df['gameId'].astype(str)
games_df['GAME_ID']    = games_df['GAME_ID'].astype(str)

# Rename NBA API columns to our standard names
boxscores_df = boxscores_df.rename(columns={
    'gameId':                   'GAME_ID',
    'personId':                 'PLAYER_ID',
    'firstName':                'FIRST_NAME',
    'familyName':               'LAST_NAME',
    'teamTricode':              'TEAM_ABBREVIATION',
    'minutes':                  'MIN',
    'points':                   'PTS',
    'reboundsTotal':            'REB',
    'assists':                  'AST',
    'steals':                   'STL',
    'blocks':                   'BLK',
    'turnovers':                'TOV',
    'fieldGoalsMade':           'FGM',
    'fieldGoalsAttempted':      'FGA',
    'fieldGoalsPercentage':     'FG_PCT',
    'threePointersMade':        'FG3M',
    'threePointersAttempted':   'FG3A',
    'threePointersPercentage':  'FG3_PCT',
    'freeThrowsMade':           'FTM',
    'freeThrowsAttempted':      'FTA',
    'freeThrowsPercentage':     'FT_PCT',
    'plusMinusPoints':          'PLUS_MINUS',
})

# Build full PLAYER_NAME from first + last
boxscores_df['PLAYER_NAME'] = boxscores_df['FIRST_NAME'] + ' ' + boxscores_df['LAST_NAME']

# Select final columns
box_cols = [
    'GAME_ID', 'PLAYER_ID', 'PLAYER_NAME', 'TEAM_ABBREVIATION',
    'MIN', 'PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV',
    'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT',
    'FTM', 'FTA', 'FT_PCT', 'PLUS_MINUS'
]
player_boxscores = boxscores_df[box_cols].copy()

# Merge in game date and season type from games table
player_boxscores = player_boxscores.merge(
    games_df[['GAME_ID', 'GAME_DATE', 'SEASON_TYPE']],
    on='GAME_ID', how='left'
)

# Cast numeric columns
num_cols = ['PTS','REB','AST','STL','BLK','TOV','FGM','FGA',
            'FG_PCT','FG3M','FG3A','FG3_PCT','FTM','FTA','FT_PCT','PLUS_MINUS']
player_boxscores[num_cols] = player_boxscores[num_cols].apply(pd.to_numeric, errors='coerce')

player_boxscores.to_csv('data/player_boxscores.csv', index=False)
print(f'PlayerBoxScores saved: {len(player_boxscores)} rows')

# Build Players table
players_df = (player_boxscores[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ABBREVIATION']]
              .drop_duplicates(subset='PLAYER_ID')
              .rename(columns={'TEAM_ABBREVIATION': 'TEAM'}))
players_df.to_csv('data/players.csv', index=False)
print(f'Players saved: {len(players_df)} unique players')

player_boxscores.head()

Found 13 chunk files — combining...
PlayerBoxScores saved: 66561 rows
Players saved: 577 unique players


,GAME_ID,PLAYER_ID,PLAYER_NAME,TEAM_ABBREVIATION,MIN,PTS,REB,AST,STL,BLK,...,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,PLUS_MINUS,GAME_DATE,SEASON_TYPE
0,22400061,1627759,Jaylen Brown,BOS,29:54,23,7,1,1,0,...,0.389,5,9,0.556,4,4,1.0,23.0,NaN,NaN
1,22400061,1628369,Jayson Tatum,BOS,30:18,37,4,10,1,1,...,0.778,8,11,0.727,1,2,0.5,26.0,NaN,NaN
2,22400061,201143,Al Horford,BOS,26:05,11,3,5,1,1,...,0.571,3,5,0.600,0,0,0.0,19.0,NaN,NaN
3,22400061,1628401,Derrick White,BOS,26:38,24,3,4,1,0,...,0.615,6,10,0.600,2,2,1.0,21.0,NaN,NaN
4,22400061,201950,Jrue Holiday,BOS,30:31,18,4,4,1,0,...,0.778,4,6,0.667,0,0,0.0,23.0,NaN,NaN


In [8]:
# ── Cell 6: Reassemble chunks into final CSVs ────────────────────────────
import glob

chunk_files = sorted(glob.glob(f"{CHECKPOINT_DIR}/chunk_*.csv"))
print(f"Found {len(chunk_files)} chunk files — combining...")

boxscores_df = pd.concat([pd.read_csv(f) for f in chunk_files], ignore_index=True)

# Drop duplicates immediately after concat
boxscores_df = boxscores_df.drop_duplicates()

# Rename NBA API columns to our standard names
boxscores_df = boxscores_df.rename(columns={
    'gameId':                   'GAME_ID',
    'personId':                 'PLAYER_ID',
    'firstName':                'FIRST_NAME',
    'familyName':               'LAST_NAME',
    'teamTricode':              'TEAM_ABBREVIATION',
    'minutes':                  'MIN',
    'points':                   'PTS',
    'reboundsTotal':            'REB',
    'assists':                  'AST',
    'steals':                   'STL',
    'blocks':                   'BLK',
    'turnovers':                'TOV',
    'fieldGoalsMade':           'FGM',
    'fieldGoalsAttempted':      'FGA',
    'fieldGoalsPercentage':     'FG_PCT',
    'threePointersMade':        'FG3M',
    'threePointersAttempted':   'FG3A',
    'threePointersPercentage':  'FG3_PCT',
    'freeThrowsMade':           'FTM',
    'freeThrowsAttempted':      'FTA',
    'freeThrowsPercentage':     'FT_PCT',
    'plusMinusPoints':          'PLUS_MINUS',
})

# Cast GAME_ID after rename — zero pad to match games_df format
boxscores_df['GAME_ID'] = boxscores_df['GAME_ID'].astype(str).str.zfill(10)
games_df['GAME_ID']     = games_df['GAME_ID'].astype(str)

# Build full PLAYER_NAME from first + last
boxscores_df['PLAYER_NAME'] = boxscores_df['FIRST_NAME'] + ' ' + boxscores_df['LAST_NAME']

# Select final columns
box_cols = [
    'GAME_ID', 'PLAYER_ID', 'PLAYER_NAME', 'TEAM_ABBREVIATION',
    'MIN', 'PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV',
    'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT',
    'FTM', 'FTA', 'FT_PCT', 'PLUS_MINUS'
]
player_boxscores = boxscores_df[box_cols].copy()

# Merge in game date and season type from games table
player_boxscores = player_boxscores.merge(
    games_df[['GAME_ID', 'GAME_DATE', 'SEASON_TYPE']],
    on='GAME_ID', how='left'
)

# Cast numeric columns
num_cols = ['PTS','REB','AST','STL','BLK','TOV','FGM','FGA',
            'FG_PCT','FG3M','FG3A','FG3_PCT','FTM','FTA','FT_PCT','PLUS_MINUS']
player_boxscores[num_cols] = player_boxscores[num_cols].apply(pd.to_numeric, errors='coerce')

player_boxscores.to_csv('data/player_boxscores.csv', index=False)
print(f'PlayerBoxScores saved: {len(player_boxscores)} rows')

# Build Players table
players_df = (player_boxscores[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ABBREVIATION']]
              .drop_duplicates(subset='PLAYER_ID')
              .rename(columns={'TEAM_ABBREVIATION': 'TEAM'}))
players_df.to_csv('data/players.csv', index=False)
print(f'Players saved: {len(players_df)} unique players')

player_boxscores.head()

Found 13 chunk files — combining...
PlayerBoxScores saved: 34928 rows
Players saved: 577 unique players


,GAME_ID,PLAYER_ID,PLAYER_NAME,TEAM_ABBREVIATION,MIN,PTS,REB,AST,STL,BLK,...,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,PLUS_MINUS,GAME_DATE,SEASON_TYPE
0,0022400061,1627759,Jaylen Brown,BOS,29:54,23,7,1,1,0,...,0.389,5,9,0.556,4,4,1.0,23.0,2024-10-22,Regular Season
1,0022400061,1628369,Jayson Tatum,BOS,30:18,37,4,10,1,1,...,0.778,8,11,0.727,1,2,0.5,26.0,2024-10-22,Regular Season
2,0022400061,201143,Al Horford,BOS,26:05,11,3,5,1,1,...,0.571,3,5,0.600,0,0,0.0,19.0,2024-10-22,Regular Season
3,0022400061,1628401,Derrick White,BOS,26:38,24,3,4,1,0,...,0.615,6,10,0.600,2,2,1.0,21.0,2024-10-22,Regular Season
4,0022400061,201950,Jrue Holiday,BOS,30:31,18,4,4,1,0,...,0.778,4,6,0.667,0,0,0.0,23.0,2024-10-22,Regular Season


In [10]:
# ── Verify a random game ─────────────────────────────────────────────────
import random

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 30)

# Ensure GAME_ID is string and zero-padded in both dataframes
player_boxscores['GAME_ID'] = player_boxscores['GAME_ID'].astype(str).str.zfill(10)
games_df['GAME_ID']         = games_df['GAME_ID'].astype(str)

# Pick a random game that exists in BOTH dataframes
valid_ids   = player_boxscores[player_boxscores['GAME_ID'].isin(games_df['GAME_ID'])]['GAME_ID'].unique()
sample_game = random.choice(valid_ids)
game_info   = games_df[games_df['GAME_ID'] == sample_game].iloc[0]

print(f"Game ID:     {sample_game}")
print(f"Date:        {game_info['GAME_DATE']}")
print(f"Matchup:     {game_info['AWAY_TEAM']} @ {game_info['HOME_TEAM']}")
print(f"Season type: {game_info['SEASON_TYPE']}")
print()

# Pull all player rows for this game
game_rows = player_boxscores[player_boxscores['GAME_ID'] == sample_game].copy()
print(f"Players found: {len(game_rows)}")
print()

# Show each team separately
for team in game_rows['TEAM_ABBREVIATION'].unique():
    team_rows = game_rows[game_rows['TEAM_ABBREVIATION'] == team]
    print(f"── {team} ──────────────────────────────")
    print(team_rows[['PLAYER_NAME','MIN','PTS','REB','AST','STL','BLK','TOV','FG_PCT','PLUS_MINUS']]
          .sort_values('PTS', ascending=False)
          .to_string(index=False, max_rows=None))
    print(f"  Team totals → PTS: {team_rows['PTS'].sum():.0f} "
          f"REB: {team_rows['REB'].sum():.0f} "
          f"AST: {team_rows['AST'].sum():.0f} "
          f"TOV: {team_rows['TOV'].sum():.0f}")
    print()

# Sanity checks
print("── Sanity checks ───────────────────────────")
teams = game_rows['TEAM_ABBREVIATION'].unique()
print(f"  ✓ Two teams present: {list(teams)}" if len(teams) == 2 else f"  ✗ Expected 2 teams, got {len(teams)}: {list(teams)}")
print(f"  ✓ GAME_DATE present: {game_info['GAME_DATE']}" if pd.notna(game_info['GAME_DATE']) else "  ✗ GAME_DATE is missing")
print(f"  ✓ SEASON_TYPE present: {game_info['SEASON_TYPE']}" if pd.notna(game_info['SEASON_TYPE']) else "  ✗ SEASON_TYPE is missing")
missing_pts = game_rows['PTS'].isna().sum()
print(f"  ✓ No missing PTS" if missing_pts == 0 else f"  ✗ {missing_pts} players missing PTS")
print(f"  ✓ Player count looks right ({len(game_rows)} players)" if 16 <= len(game_rows) <= 30 else f"  ✗ Unusual player count: {len(game_rows)}")

Game ID:     0022400049
Date:        2024-11-29
Matchup:     SAC @ POR
Season type: Regular Season

Players found: 30

── POR ──────────────────────────────
     PLAYER_NAME   MIN  PTS  REB  AST  STL  BLK  TOV  FG_PCT  PLUS_MINUS
   Deandre Ayton 40:57   26    9    2    1    1    1   0.579         7.0
 Anfernee Simons 40:43   21    5    9    2    0    4   0.571         9.0
     Deni Avdija 36:40   20    9    5    2    1    0   0.467        12.0
   Dalano Banton 23:17   17    2    2    5    0    4   0.538        11.0
  Shaedon Sharpe 37:29   14    4    5    0    0    1   0.400         2.0
  Toumani Camara 32:45   11    4    2    0    1    0   0.500        -1.0
   Jabari Walker  9:42    4    4    1    0    0    0   1.000         7.0
     Kris Murray 15:41    2    3    1    0    0    1   0.250        -3.0
      Duop Reath  2:46    0    0    0    0    0    0   0.000         1.0
 Donovan Clingan   NaN    0    0    0    0    0    0   0.000         0.0
    Jerami Grant   NaN    0    0    0   